# Initial Setup

This notebook begins by fetching the test data out of reframe. It then parses it for performance metrics, and finally runs a two-sample t-test on the data to determine if any variance is statistically significant.

The next cell allows you to set some basic variables to control this.

In [ ]:
# What session extra data filtering should reframe use to locate the baseline test?
tag_baseline = "test==\"no_monitoring\" and (\"jobsize\" not in locals() or jobsize==\"E\")" #and jobsize==\"D\"" # and (\"jobsize\" not in locals() or jobsize==\"E\")

# What session extra data filtering should it use to locate the various scenarios?
# This is a dict of name: value, where name will be used for later labeling of results
tags_varing = { 
    "HPCPerfStats": "test==\"hpcperfstats\"  and (\"jobsize\" not in locals() or jobsize==\"E\")",
    "LDMS": "test==\"ldms\" and (\"jobsize\" not in locals() or jobsize==\"E\")",
    "BMC_with_helper": "test==\"bmc_with_amsd_clean\" and (\"jobsize\" not in locals() or jobsize==\"E\")",
    "BMC_no_helper": "test==\"bmc_no_amsd\" and (\"jobsize\" not in locals() or jobsize==\"E\")",
}

# If needed, this can be used to limit the date range
date_range = "" # for example: now-1d:now

# If needed, any extra arguments for reframe
reframe_extra_args = ""

# Acceptable variation from the baseline to be considered equivalent, percent
equivalent_baseline = 0.005 # 0.5%

Begin by loading and paring the reframe data for each scenario:

In [ ]:
from json import loads

json={}
tagsets = { "baseline": tag_baseline } | tags_varing

for key in tagsets:
    # Call reframe to get the JSON representation of all of our baseline jobs.
    jres = ! ./reframe.sh {reframe_extra_args} --describe-stored-sessions '{date_range}?{tagsets[key]}'
    json[key] = loads('\n'.join(jres)) # Join it into one string

from pprint import pprint
pprint(json)

And transform that into a data structure that makes it easy to do comparisons.

In [ ]:
import re

perf = {} # Scenario, then name of test, then performance metric, containing a list of test results
metric_name_re = re.compile(r'[^:]+:[^:]+:([^:]+)')

for scenario in json:
    perf[scenario] = {}
    for session in json[scenario]:
        for run in session['runs']:
            for testcase in run['testcases']:
                if testcase['fail_phase'] is None:
                    testname = testcase['name']
                    if testname not in perf[scenario]:
                        perf[scenario][testname] = {}
                    for metric in testcase['perfvalues']:
                        # Do some string processing to get the name of the perf counter
                        metric_name = metric_name_re.match(metric).group(1)
                        # Check if that counter already exists and add if not
                        if metric_name not in perf[scenario][testname]:
                            perf[scenario][testname][metric_name] = {
                                "values": [],
                                "unit": testcase['perfvalues'][metric][4]
                            }
                        perf[scenario][testname][metric_name]['values'].append(
                            testcase['perfvalues'][metric][0]
                        )
pprint(perf)

And finally, run a group of t-Tests.

In [ ]:
from scipy import stats
import numpy

scenarios = list(perf.keys())
scenarios.remove('baseline')

for scenario in scenarios:
    print(perf['baseline'].keys())
    for test in perf['baseline'].keys():
        for metric in perf['baseline'][test].keys():
            basedata = perf['baseline'][test][metric]['values']
            exprdata = perf[scenario][test][metric]['values']
            res_neq = stats.ttest_ind(
                basedata,
                exprdata,
                equal_var=False
            )
            bound = numpy.mean(basedata) * equivalent_baseline
            res_eq_low = stats.ttest_ind(
                exprdata,
                basedata - bound,
                alternative='greater',
                equal_var=False
            )
            res_eq_high = stats.ttest_ind(
                exprdata,
                basedata + bound,
                alternative='less',
                equal_var=False
            )
            res_eq_p = max(res_eq_low.pvalue, res_eq_high.pvalue)
            print(f"""{scenario}: {test}: {metric}: {len(exprdata)}/{len(basedata)} experimental/base samples
    t={res_neq.statistic} 
    p-value, different={res_neq.pvalue}
    p-value, equivalent={res_eq_p} (lower: {res_eq_low.pvalue}, upper: {res_eq_high.pvalue})
    baseline:     mean: {numpy.mean(basedata)}, stddev: {numpy.std(basedata)}
    experimental: mean: {numpy.mean(exprdata)}, stddev: {numpy.std(exprdata)}
    difference: {((numpy.mean(exprdata)-numpy.mean(basedata))/numpy.mean(basedata))*100}%""")